# 06 - Explainability (bonus)

| Part | Method |
|---|---|
| Global | top TF-IDF features per class by logistic regression coefficient |
| Local | per-text feature contributions (`tfidf value x coefficient`) for correct and wrong predictions |

- Model: reference TF-IDF + LogReg from 04 (same subsample, same config)
- For a linear model the contribution of each n-gram to a class score is exact, no approximation as in LIME / SHAP
- Kaggle: CPU is enough, Internet on, competition data attached

## Setup

In [ ]:
import subprocess
import sys

REPO = "https://github.com/mohinska/ai_hw3_AIGenTextDetection.git"
REPO_DIR = "/tmp/repo"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

import common
from common import LABELS, TEXT_COL, LABEL_COL, SEED
from features import fit_tfidf, tfidf_feature_names, tfidf_transform

warnings.filterwarnings("ignore", category=UserWarning)

## Config

In [ ]:
N_TRAIN = 60_000
REF_FEATURES = ["char_wb_2-5"]             # best feature set from 02
REF_C = 2.0                                 # best C from 02
TOP_K = 15

## Fit the reference model

In [ ]:
train_df, _, _ = common.load_data()
trn, val = common.split_train_val(train_df)
trn = common.subsample(trn, N_TRAIN)
y_val = val[LABEL_COL].to_numpy()

vecs = fit_tfidf(REF_FEATURES, trn[TEXT_COL].fillna(""))
X_trn = tfidf_transform(vecs, trn[TEXT_COL].fillna(""))
X_val = tfidf_transform(vecs, val[TEXT_COL].fillna(""))
clf = LogisticRegression(C=REF_C, max_iter=2000, random_state=SEED).fit(X_trn, trn[LABEL_COL])
val_pred = clf.predict(X_val)
feat = tfidf_feature_names(vecs)
CLS = {c: i for i, c in enumerate(clf.classes_)}
print(f"val macro-F1: {f1_score(y_val, val_pred, average='macro'):.4f} | features: {len(feat):,}")

## Global: features with the largest coefficients per class

- `word_1-2:` = word / word-pair, `char_wb_2-5:` = character n-gram (`_` marks a space)

In [ ]:
def top_features(cls, k=TOP_K, prefix=None):
    coef = clf.coef_[CLS[cls]]
    mask = np.char.startswith(feat.astype(str), prefix) if prefix else np.ones(len(feat), bool)
    idx = np.where(mask)[0][np.argsort(-coef[mask])[:k]]
    return [(feat[i].split(":", 1)[1].replace(" ", "_"), round(float(coef[i]), 2)) for i in idx]


for prefix in [n + ":" for n in REF_FEATURES]:
    print(prefix)
    display(pd.DataFrame({c: [f"{f} ({w})" for f, w in top_features(c, prefix=prefix)] for c in LABELS}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, c in zip(axes, LABELS):
    names_, weights = zip(*top_features(c, k=12, prefix="word_1-2:"))
    ax.barh(names_[::-1], weights[::-1], color={"human": "tab:blue", "ai": "tab:red", "fusion": "tab:green"}[c])
    ax.set_title(f"{c}: top word features")
common.save_fig("xai_top_words.png")

## Local: why a given text got its label

- Contribution of feature j to class k = `tfidf_j(text) x coef_kj`
- Shown: one correct prediction per class and the most common error (true `fusion`, predicted `human`)

In [ ]:
def explain(i, k=10):
    row = X_val[i]
    pred = val_pred[i]
    contrib = row.multiply(clf.coef_[CLS[pred]]).tocsr()
    order = np.argsort(-contrib.data)[:k]
    text = textwrap.shorten(val[TEXT_COL].iloc[i], 300)
    print(f"--- true: {y_val[i]} | pred: {pred} | domain: {val['domain'].iloc[i]}\n{text}")
    display(pd.DataFrame({"feature": [feat[contrib.indices[j]].split(":", 1)[1].replace(" ", "_") for j in order],
                          "contribution": np.round(contrib.data[order], 3)}).T)


rng = np.random.default_rng(SEED)
cases = [(c, c) for c in LABELS] + [("fusion", "human")]
for true, pred in cases:
    idx = np.where((y_val == true) & (val_pred == pred))[0]
    if len(idx):
        explain(rng.choice(idx))

## Conclusions

| Finding | Decision |
|---|---|
| <!-- TODO --> | <!-- TODO --> |
| <!-- TODO --> | <!-- TODO --> |